# Assignment 2: Translation with LLM
**Author:** Sayan H. Mandal  
**Model:** Helsinki-NLP/opus-mt-en-hi (HuggingFace)  
**Metrics:** BLEU, CHRF, TER

In [ ]:
# STEP 0: Install dependencies
!pip install -q transformers sentencepiece sacrebleu pandas openpyxl torch

In [ ]:
# STEP 1: Upload assignment1_output.xlsx
from google.colab import files
uploaded = files.upload()  # Upload assignment1_output.xlsx when prompted

In [ ]:
import re
import pandas as pd
import sacrebleu
from transformers import MarianMTModel, MarianTokenizer
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# STEP 2: Load 100 sentences
df = pd.read_excel('assignment1_output.xlsx')
df.columns = ['English Sentences', 'Hindi Sentences',
              'Word Count (English)', 'Word Count (Hindi)', 'Difference']
sample = df.sample(n=100, random_state=42).reset_index(drop=True)
english_sentences = sample['English Sentences'].tolist()
reference_hindi   = sample['Hindi Sentences'].tolist()
print(f'Loaded 100 sentences')

In [ ]:
# STEP 3: Load model
MODEL_NAME = 'Helsinki-NLP/opus-mt-en-hi'
tokenizer  = MarianTokenizer.from_pretrained(MODEL_NAME)
model      = MarianMTModel.from_pretrained(MODEL_NAME)
print('Model loaded!')

In [ ]:
# STEP 4: Translate
def translate(sentences, tokenizer, model, batch_size=16):
    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt',
                           padding=True, truncation=True, max_length=512)
        outputs = model.generate(**inputs, num_beams=4, max_length=512)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
        print(f'Translated {min(i+batch_size, len(sentences))}/100...')
    return translations

predicted_hindi = translate(english_sentences, tokenizer, model)
print('Translation complete!')

In [ ]:
# STEP 5: Compute Metrics
refs = [reference_hindi]
bleu = sacrebleu.corpus_bleu(predicted_hindi, refs).score
chrf = sacrebleu.corpus_chrf(predicted_hindi, refs).score
ter  = sacrebleu.corpus_ter(predicted_hindi, refs).score

print(f'BLEU : {bleu:.4f}')
print(f'CHRF : {chrf:.4f}')
print(f'TER  : {ter:.4f}')

In [ ]:
# STEP 6: Save metrics to .txt
with open('assignment2_metrics.txt', 'w', encoding='utf-8') as f:
    f.write('=' * 50 + '\n')
    f.write('Assignment 2: Translation Evaluation Metrics\n')
    f.write('=' * 50 + '\n\n')
    f.write('Model Used  : Helsinki-NLP/opus-mt-en-hi (HuggingFace)\n')
    f.write('Sentences   : 100 (sampled from Assignment 1 output)\n\n')
    f.write(f'BLEU Score  : {bleu:.4f}\n')
    f.write(f'CHRF Score  : {chrf:.4f}\n')
    f.write(f'TER Score   : {ter:.4f}\n\n')
    f.write('Metric Notes:\n')
    f.write('  BLEU : Higher is better (0-100). N-gram overlap precision.\n')
    f.write('  CHRF : Higher is better (0-100). Character n-gram F-score.\n')
    f.write('  TER  : Lower is better. Minimum edits to match reference.\n')
print('Metrics saved!')

In [ ]:
# STEP 7: Save translations to Excel
def clean_text(text):
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', str(text))

wb = Workbook()
ws = wb.active
ws.title = 'Translations'

headers     = ['Original English Sentence', 'Model-Generated Hindi Translation']
header_font = Font(bold=True, name='Arial', color='FFFFFF', size=11)
header_fill = PatternFill('solid', start_color='4472C4', end_color='4472C4')
header_aln  = Alignment(horizontal='center', vertical='center', wrap_text=True)

for col_idx, col_name in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.font = header_font; cell.fill = header_fill; cell.alignment = header_aln

for row_idx, (eng, hi) in enumerate(zip(english_sentences, predicted_hindi), start=2):
    ws.cell(row=row_idx, column=1, value=clean_text(eng)).font = Font(name='Arial', size=10)
    ws.cell(row=row_idx, column=2, value=clean_text(hi)).font  = Font(name='Arial', size=10)

ws.column_dimensions['A'].width = 70
ws.column_dimensions['B'].width = 70
ws.freeze_panes = 'A2'

wb.save('assignment2_translations.xlsx')
print('Excel saved!')

In [ ]:
# STEP 8: Download both output files
from google.colab import files
files.download('assignment2_translations.xlsx')
files.download('assignment2_metrics.txt')
print('Done! Both files downloaded.')